# 08 - Simulation Intro


This notebook introduces the `e.simulation` namespace with one measured Ar fit, one measured 2.8 M PhOH EEC' fit, and a simulated scan-rate sweep from the fitted PhOH model. Group fitting across multiple CVs now lives in `09_group_fitting.ipynb`.


## Import eCAT And Set Paths

Simulation examples use the packaged Fe/PhOH CV data for fit-ready measured-current inputs.


In [ ]:
from copy import deepcopy
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def display_path(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return path.name

import ecat as e

DATA_DIR = ROOT / "examples" / "data" / "fe_phoh_cv"
EXPORT_DIR = ROOT / "notebooks" / "_outputs"
EXPORT_DIR.mkdir(exist_ok=True)

e.plotting_style("notebook")
print("eCAT version:", getattr(e, "__version__", "unknown"))
print("Example data:", display_path(DATA_DIR))
print("Text files:", len(list(DATA_DIR.glob("*.txt"))))


## Check Optional Simulation Backend

The simulation module is designed to stay import-safe even when the optional ElectroKitty backend is not installed. You can install into the current notebook kernel with `%pip install "ecat[simulation]"` or `%pip install electrokitty`; after installation, restart the kernel if imports still fail.


In [ ]:
import importlib.util

HAS_ELECTROKITTY = importlib.util.find_spec("electrokitty") is not None
print("ElectroKitty available:", HAS_ELECTROKITTY)
if not HAS_ELECTROKITTY:
    raise ImportError(
        'This supported simulation tutorial needs the optional backend. '
        'Run %pip install "ecat[simulation]", restart the kernel, and rerun the notebook.'
    )

## Load And Select Real CV Inputs

Use eCAT filters so the simulation inputs are traceable. Ar CVs use the `0` to `-1.5 V` fitting window; PhOH CVs use expanded waveform trimming from `-1` to `-1.7 V` before fitting and simulation.


In [ ]:
cvs = e.get_data({
    "folder path": str(DATA_DIR),
    "reference mode": "keyword",
    "reference keyword": "Fc",
    "reference guess": 0.4,
    "reference label": "Fc/Fc+",
    "electrode diameter": 0.3,
    "print": False,
})
cvs = e.filter(cvs, {"segments": 3}, {"print": False})

fe_ar = e.filter(cvs, {
    "gas": "Ar",
    "compounds": "Fe-tpyPY2Me",
}, {"logic": "AND", "print": False})

phoh_co2 = e.filter(cvs, {
    "gas": "CO2",
    "compounds": "PhOH",
}, {"logic": "AND", "print": False})

scan_series = e.filter(fe_ar, {
    "scan window": [-1.7, 1],
}, {"print": False})
scan_series = e.sort(scan_series, "scan rate", {"print": False})

ar_reference_cv = e.filter(scan_series, {"scan rate": 0.1}, {"print": False})[0]
phoh_group = e.filter(phoh_co2, {"scan rate": 0.1}, {"print": False})
phoh_28m_cv = e.filter(phoh_group, {"species": "2.8M PhOH"}, {"print": False})[0]

AR_CV_POTENTIAL_WINDOW = [-0.7, -1.7]
PHOH_CV_POTENTIAL_WINDOW = [-1.0, -1.7]
AR_CV_WINDOW_OPTIONS = {
    "potential window": AR_CV_POTENTIAL_WINDOW,
    "trim mode": "expand",
}
PHOH_CV_WINDOW_OPTIONS = {
    "potential window": PHOH_CV_POTENTIAL_WINDOW,
    "trim mode": "expand",
}

e.show_objects([ar_reference_cv, phoh_28m_cv], {
    "columns": ["gas", "scan rate", "compounds", "scan window"],
});


## Build A Programmatic CV Input

`cv_program()` returns a `SimulatedCVInput`. Use `.show()` for a setup table and `.plot()` to inspect the potential program. Quiet time is metadata and is drawn at negative time only when `plot quiet time` is enabled.


In [ ]:
program = e.simulation.cv_program(
    Ei=0.0,
    E_low=-1.5,
    scan_rate=0.1,
    segments=2,
    points_per_segment=300,
    quiet_time=5,
    incubation_time=0,
)
program.show()
program.plot({"plot quiet time": True, "label": "Program + quiet time"});

## Convert A Real CV With `cv_data`

`cv_data()` makes a fit-ready simulation input from an eCAT `cv`. A larger `stride` keeps the tutorial quick while preserving the measured-current path.


In [ ]:
input_from_data = e.simulation.cv_data(phoh_28m_cv, {
    **PHOH_CV_WINDOW_OPTIONS,
    "stride": 20,
})
input_from_data.show()
input_from_data.plot()


## Simulation Unit Conventions

Simulation parameters use SI-derived public units: potentials in `V`, time in `s`, current in `A`, scan rate in `V/s`, bulk concentrations in `mol/m^3`, surface coverages in `mol/m^2`, diffusion in `m^2/s`, area in `m^2`, resistance in `ohm`, temperature in `K`, and `cell.Cdl` as total capacitance in `F`. `cell.Cdl='auto'` estimates total farads from measured forward/reverse current separation; eCAT converts to backend area-normalized capacitance internally when a backend needs it. Do not divide by electrode area before passing `cell.Cdl`.


## Backend Simulation

Run a simple simulated CV from the programmatic input. This requires the optional ElectroKitty simulation dependency; if the check cell above reports `False`, install it in the notebook kernel before running the rest of the simulation examples.


In [ ]:
params = {
    "concentrations": {"bulk": {"a": 1.0, "b": 0.0}},
    "diffusion": {"a": 1e-9, "b": 1e-9},
    "kinetics": [{"E0": -0.8, "k0": 1e-3, "alpha": 0.5}],
    "cell": {"T": 298.15, "Ru": 0.0, "Cdl": 0.0, "A": 1e-5},
    "spatial": "fast",
}

result = e.simulation.simulate_cv(
    program,
    "E",
    params,
    options={"plot": True, "check params": True},
)
result.show({"print setup": True, "print params": True})

## Pre-Equilibrium And Chemical Incubation

Reaction-local `K` values equilibrate the entered concentrations before the run. A reversible reaction with `equilibrate=False` remains dynamic but is omitted from that initial algebraic solve. `incubation_time` then evolves bulk homogeneous chemical reactions for the requested time before the backend quiet-time hold and potential scan; surface and mixed-phase steps remain backend-only. It defaults to `0 s` and does not run electron transfer, diffusion, Ru, Cdl, or the potential program.

No pool declaration is needed. The reaction stoichiometry supplies the conservation constraints. `print states` shows only species that changed across the entered, equilibrated, and incubated states.

In [ ]:
incubation_program = program.with_incubation_time(4.0)
incubation_mechanism = "E(1):Ox=Red\nC:Red=Resting\nC:Resting>Product"
incubation_params = {
    "concentrations": {
        "bulk": {"Ox": 0.0, "Red": 1.0, "Resting": 0.0, "Product": 0.0},
    },
    "diffusion": {name: 1e-9 for name in ["Ox", "Red", "Resting", "Product"]},
    "kinetics": [{"E0": -0.8, "k0": 1e-3, "alpha": 0.5}],
    "reactions": {
        "Red=Resting": {"K": 4.0, "k_exchange": 10.0},
        "Resting>Product": {"k": 0.25},
    },
    "cell": {"T": 298.15, "Ru": 0.0, "Cdl": 0.0, "A": 1e-5},
    "spatial": "fast",
}

incubation_result = e.simulation.simulate_cv(
    incubation_program,
    incubation_mechanism,
    incubation_params,
    options={"plot": False, "print setup": False, "print params": False},
)
incubation_result.show({
    "print setup": True,
    "print params": "compact",
    "print states": True,
})

## Useful Simulation String Inputs

The fit examples below use a few string conveniences from the simulation test notebook. The spatial `fast` preset is used here for speed so the notebook is easy to rerun; use `balanced` or `accurate` for more careful production simulations.


In [ ]:
def wrap_df(df, value_width="220px", meaning_width="360px"):
    return df.style.set_properties(
        **{"white-space": "normal", "text-align": "left"}
    ).set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("max-width", value_width)]},
        {"selector": "td.col2", "props": [("max-width", meaning_width)]},
    ])

simulation_string_inputs = pd.DataFrame([
    {"Where": "fit['vary']", "String": "E0_0", "Meaning": "formal potential for electron-transfer step 0"},
    {"Where": "fit['vary']", "String": "E0_1", "Meaning": "formal potential for electron-transfer step 1"},
    {"Where": "fit['vary']", "String": "D", "Meaning": "one tied diffusion coefficient when all diffusion values start equal"},
    {"Where": "fit['fixed']", "String": "alpha_*", "Meaning": "all kinetic alpha entries"},
    {"Where": "fit['fixed']", "String": "k0_*", "Meaning": "all heterogeneous electron-transfer rate constants"},
    {"Where": "params['kinetics'][...]['k0']", "String": "fast, reversible, rev", "Meaning": "k0 aliases that normalize to 1e-3 m/s"},
    {"Where": "params['kinetics'][...]['k0']", "String": "quasi, quasireversible, quasi reversible", "Meaning": "k0 aliases that normalize to 1e-5 m/s"},
    {"Where": "params['kinetics'][...]['k0']", "String": "slow, irreversible, irrev", "Meaning": "k0 aliases that normalize to 1e-8 m/s"},
    {"Where": "params['spatial']", "String": "fast, balanced, accurate", "Meaning": "spatial grid presets (see below)"},
    {"Where": "options", "String": "post correction = offset", "Meaning": "apply only a final vertical current offset after fitting"},
])

spatial_presets = (
    pd.DataFrame.from_dict(e.simulation.SPATIAL_PRESETS, orient="index")
    .rename_axis("preset")
    .reset_index()
    .rename(columns={
        "dx_fraction": "dx_fraction / dimensionless",
        "viscosity": "viscosity / m<sup>2</sup> s<sup>-1</sup>",
        "rotation": "rotation / Hz",
    })
)
spatial_presets["relative grid cost / fast=1"] = (
    spatial_presets["nx"] / spatial_presets["dx_fraction / dimensionless"]
)
spatial_presets["relative grid cost / fast=1"] /= spatial_presets.loc[
    spatial_presets["preset"] == "fast", "relative grid cost / fast=1"
].iloc[0]

spatial_style = (
    spatial_presets.style
    .format({
        "dx_fraction / dimensionless": "{:.3e}",
        "viscosity / m<sup>2</sup> s<sup>-1</sup>": "{:.3e}",
        "rotation / Hz": "{:.3e}",
        "relative grid cost / fast=1": "{:.1f}x",
    })
    .format_index(escape=None, axis=1)
)

display(wrap_df(simulation_string_inputs))
display(spatial_style)


## Fit 1: Single Ar CV

Fit one Ar CV at `0.1 V/s` to get an `EE` starting point for the formal potentials and tied diffusion coefficient.


In [ ]:
AR_FIT_STRIDE = 15
CO2_FIT_STRIDE = 15

ar_ee_params = {
    "concentrations": {"bulk": {"FeII": 1.0, "FeI": 0.0, "Fe0": 0.0}},
    "diffusion": {"FeII": 2e-9, "FeI": 2e-9, "Fe0": 2e-9},
    "kinetics": [
        {"E0": -1.3, "k0": 1e-3, "alpha": 0.5},
        {"E0": -1.5, "k0": 1e-3, "alpha": 0.5},
    ],
    "cell": {"T": 298.15, "Ru": 0.0, "Cdl": "auto", "A": 1e-5},
    "spatial": "fast",
}

single_ar_fit = e.simulation.fit_cv(
    ar_reference_cv,
    "EE",
    ar_ee_params,
    fit={
        "vary": ["E0_0", "E0_1", "D"],
        "fixed": {"alpha_*": 0.5, "k0_*": 1e-3},
        "bounds": "auto",
        "transform": "auto",
    },
    options={
        "cv data": {**AR_CV_WINDOW_OPTIONS, "stride": AR_FIT_STRIDE, "estimate Cdl": "auto"},
        "plot": True,
        "post correction": "offset",
        "print stats": False,
        "print corrections": False,
        "print progress": False,
    },
)
single_ar_fit.show({"print setup": False, "print stats": True, "print corrections": True, "print params": False, "print simulation": False})

## Fit 2: One 2.8 M PhOH EEC' CV

Use the Ar-fit parameters as the starting point. The PhOH concentration is mapped to the simulation species `Substrate`, and only the catalytic forward rate is varied in this tutorial fit.


In [ ]:
eecat_params = deepcopy(single_ar_fit.best_params)
eecat_params.setdefault("concentrations", {}).setdefault("bulk", {}).update({
    "Substrate": 2800.0,
    "Product": 0.0,
})
reference_diffusion = next(iter(eecat_params.get("diffusion", {"FeII": 1e-9}).values()))
eecat_params.setdefault("diffusion", {}).update({
    "Substrate": reference_diffusion,
    "Product": reference_diffusion,
})
eecat_params["reactions"] = [{"kf": 1.0, "kb": 0.0}]
eecat_params["cell"] = "auto"

eecat_mechanism = (
    "E(1):FeII=FeI\n"
    "E(1):FeI=Fe0\n"
    "C:Fe0+Substrate>FeI+Product"
)

phoh_28m_fit = e.simulation.fit_cv(
    phoh_28m_cv,
    eecat_mechanism,
    eecat_params,
    fit={
        "vary": ["reactions.0.kf"],
        "fixed": {"alpha_*": 0.5, "k0_*": 1e-3},
        "bounds": "auto",
        "transform": "auto",
    },
    options={
        "cv data": {**PHOH_CV_WINDOW_OPTIONS, "stride": CO2_FIT_STRIDE, "estimate Cdl": "auto"},
        "concentration mapping": {"PhOH": "Substrate"},
        "plot": True,
        "post correction": "offset",
        "print stats": False,
        "print corrections": False,
        "print progress": False,
    },
)
phoh_28m_fit.show({"print setup": False, "print stats": True, "print corrections": True, "print params": True, "print simulation": False})

## Simulated EEC' Scan-Rate Sweep

Use the fitted 2.8 M PhOH result as a template and rerun the same simulated CV input at several scan rates. This is a simulation-only sweep; group fitting measured scan-rate series is covered in `09_group_fitting.ipynb`.


In [ ]:
EECAT_SCAN_RATES = [0.025, 0.05, 0.1, 0.25, 0.5, 1.0]

scan_rate_results = [
    phoh_28m_fit.simulation_result.with_scan_rate(scan_rate, options={"plot": False})
    for scan_rate in EECAT_SCAN_RATES
]
scan_rate_labels = [f"{scan_rate:g} V/s" for scan_rate in EECAT_SCAN_RATES]

ax = e.multiplot(scan_rate_results, {
    "labels": scan_rate_labels,
    "title": "EEC' Scan-Rate Sweep From 2.8 M PhOH Fit",
    "print": False,
})

rows = []
for scan_rate, result in zip(EECAT_SCAN_RATES, scan_rate_results):
    rows.append({
        "scan rate / V s^-1": scan_rate,
        "points": len(result.data),
        "current min / A": result.data["Current"].min(),
        "current max / A": result.data["Current"].max(),
    })

display(pd.DataFrame(rows))

## simulation.fit_cv Options

Use `describe_options()` to inspect the notebook-facing fitting options. The short `fit_cv` name resolves to `simulation.fit_cv`.


In [ ]:
e.describe_options("fit_cv")